# Model ENSO full 12-2022-holdconstant

December 2022, January 2023, February 2025

Caroline Juang, c.juang@columbia.edu

**Inputs:**
* SST gradient (1982-present) - build it following the README
* trained models `Model_ENSOclim_Akaike` (SST gradient -> climate)
* trained models `Model_Akaike` (climate -> burned area)

**Variables**
* `sstpred` generally refers to the detrended SST gradient, which is the detrended-SST gradient scenario put into the climate->burned area model.
* `climpred` generally refers to the observed SST gradient, put into the climate->burned area model.

**Modified model**
* Replace parts of the trained model with constants, for these experiments:
* 1: warming variables only
    * forest: hold all non-Tmax and non-VPD variables constant.
    * non-forest: hold all non-Tmax, non-VPD, and non-current year prec constant.
* 2: prior wetting vars only
    * forest and non-forest: hold all but non- prior year prec constant.
* 3: y0 wetting variables only
    * forest and non-forest: hold all but current year prec constant.

**Outputs:**
* Predicted climate (1983-present) (but only for the relevant variables to the climate->burned area model)
* Predicted burned area (1984-present) from the trended and detrended scenarios into the climate->burned area model.
* Predicted burned area, detrended. `wumiDT`.

This model will predict burned area in ecoregions in the western US using seasonal SST gradient values.

The model is based on the outputs of `Model_ENSOclim_Akaike` and `Model_Akaike`.

Data source:
* NOAA sea surface temperature, https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html

In [1]:
# import
from customconfig import *
from customscripts import *

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import joblib

In [2]:
# customize seasons for climate variables

time_length = int(finalyear-firstyear+1) # get length of timeseries
# translate years into dates
firsttime = str(firstyear)+'-01-01'
finaltime = str(finalyear)+'-12-31'

# Read the climate index from the .txt file
with open('0_climindname.txt', 'r') as f:
    climindname = f.read().strip()
print(climindname +' will be used for the SST gradient')
climindnameDT = 'DeTrend_'+climindname
climfilename = 'DeTrendClimObs_'+climindname # sst-predicted BA

# importing data string
data_string = 'data//'
model_string = 'model//'+climindname+'//'
predict_string = 'predicted//'+climindname+'//'

# folder for saving figures
figfolder = 'your_figures_folder' # customize this

patch125-155_nino3-34 will be used for the SST gradient


In [3]:
# manual hold variables constant (2/10/2025)

# REMOVE ALL BUT y0 TMAX AND VPD VARS (for forest)
clim_warmingfor = [
                 'solar y0', 'wind y0', 'tmin y0', 'wetdays y0',
                 'rh y0', 'rh y-1', 'prec y0', 'prec y-1',
                 'wind y-1', 'tmax y-1','tmin y-1',
                 'wetdays y-1','solar y-1','vpd y-1']
# REMOVE ALL but y0 TMAX, VPD, PREC (for non-forest)
clim_warmingnon = [
                 'solar y0', 'wind y0', 'tmin y0', 'wetdays y0',
                 'rh y0', 'rh y-1', 'prec y-1',
                 'wind y-1', 'tmax y-1','tmin y-1',
                 'wetdays y-1','solar y-1','vpd y-1']

# REMOVE ALL BUT y-1 PREC
clim_priorwetting = [
                 'solar y0', 'wind y0', 'tmax y0', 'tmin y0', 'wetdays y0',
                 'rh y0', 'rh y-1', 'prec y0', 'vpd y0',
                 'wind y-1', 'tmax y-1','tmin y-1',
                 'wetdays y-1','solar y-1','vpd y-1']
# REMOVE ALL BUT y-0 PREC
clim_currwetting = [
                 'solar y0', 'wind y0', 'tmax y0', 'tmin y0', 
                 'wetdays y0', 'rh y0', 'vpd y0',
                 'rh y-1', 'prec y-1',
                 'wind y-1', 'tmax y-1','tmin y-1',
                 'wetdays y-1','solar y-1','vpd y-1']

# compile these experiments into a bigger df, with their names
experimentnames = ['warmingvarsonly', 'prioryrwettingonly', 'y0wettingonly']
experimentfor_list = [clim_warmingfor, clim_priorwetting, clim_currwetting]
experimentnon_list = [clim_warmingnon, clim_priorwetting, clim_currwetting]

## scripts

In [4]:
# opposite of log10
def exponentialit(value):
    return 10**value

In [5]:
# average climate variables, within a selected season

def annual_seasonAvg(data, firstmonth, finalmonth):
    """
    This intakes an array of current ecoregion's climate variable of monthly averages (data), 
    Output: an array of yearly averages of the ecoregion climate variable, in the
    seasons specified (firstmonth, finalmonth).
    Requirements: time = an xarray timeseries of months in datetime format.
    """
    withinyear = (time['time.year']>= years[0]) & (time['time.year'] <= years[-1])
    withinseason = (time['time.month'] >= firstmonth) & (time['time.month'] <= finalmonth)
    thistime = time[withinseason & withinyear] # cut time

    # create pd dataframe based on data, for time resampling
    thisdf = pd.DataFrame({'time': pd.to_datetime(thistime.values), 'clim':data[withinyear & withinseason]}).set_index('time')
    thisdf = thisdf.resample('Y').mean().reset_index()
    return thisdf.clim.values

In [6]:
# just format the name of the climate variable
def suppnameformat(plotvarname):
    """
    Input is `plotvarname`, which is the climate name as it is formatted in the 
    pandas dataframe, but just the name and no month or year. (e.g. prec, vpd, rh)
    Reformat the name of the climate plus include units.
    Returns [fullname, timeseries, units]
    """
    plotvarname = plotvarname.capitalize()
    # color palette: "omni spring pastels" from https://www.heavy.ai/blog/12-color-palettes-for-telling-better-stories-with-your-data
    colorpalette = ["#fd7f6f", "#7eb0d5", "#b2e061", "#bd7ebe", "#ffb55a", "#ffee65", "#beb9db", "#fdcce5", "#8bd3c7"]
    # fix names, assign units and color
    if 'Vpd' in plotvarname:
        plotvarname = 'VPD'
        units = 'hPa'
        color = colorpalette[0]
    elif 'Prec' in plotvarname:
        plotvarname = 'Precipitation'
        units = 'mm'
        color = colorpalette[1]
    elif 'Rh' in plotvarname:
        plotvarname = 'RH'
        units = '%'
        color = colorpalette[2]
    elif 'Wetdays' in plotvarname:
        plotvarname = 'Wet_days'
        units = 'fraction of days >2.54 mm'
        color = colorpalette[3]
    elif 'Solar' in plotvarname:
        plotvarname = 'Solar_radiation'
        units = 'W/m$^2$'
        color = colorpalette[4]
    elif 'Tmax' in plotvarname:
        units = '$\degree$C'
        color = colorpalette[5]
    elif 'Tmin' in plotvarname:
        units = '$\degree$C'
        color = colorpalette[6]
    elif 'Wind' in plotvarname:
        units = 'm/s'
        color = colorpalette[7]
    elif 'Gsst' in plotvarname:
        plotvarname = 'gSST'
        units = '$\degree$C'
        color = 'k'
    return plotvarname, units, color

In [7]:
# format plot script
def suppclimplotformat(variablename):
    """
    Input is `variablename`, which is the climate name as it is formatted in the pandas dataframes
    (e.g. prec y0 mo 1-3, or "variable_name year month month-numbers"). 
    Reformat the name of the plot, and also provide the unit labels and timeseries 
    for the y-axis of the plot.
    Returns [fullname, timeseries, units] (e.g. Precipitation y-0 JFM, 1984-2022, mm)
    """
    varsplit = variablename.split(' ')
    plotvaryear = varsplit[1] # y0 or y-1
    plotvarmo = varsplit[3] # month numbers
    # fix names
    plotvarname, units, color = suppnameformat(varsplit[0])

    # fix months
    if '1-3' in varsplit[3]:
        plotvarmo = 'JFM'
    elif '4-6' in varsplit[3]:
        plotvarmo = 'AMJ'
    elif '7-9' in varsplit[3]:
        plotvarmo = 'JAS'
    elif '10-12' in varsplit[3]:
        plotvarmo = 'OND'

    # fix years
    if 'y0' in varsplit[1]:
        timeseries = np.arange(firstyear, finalyear+1)
        plotvaryear = 'y-0'
    elif 'y-1' in varsplit[1]:
        timeseries = np.arange(firstyear-1, finalyear)

    fullname = plotvarname + ' ' + plotvaryear + ',' + plotvarmo
    return fullname, timeseries, units

In [8]:
# printing significance
def addSigMarker(pvalue):
    """
    Add an asterisk for values that are significant.
    p<0.05, add *
    p<0.01, add **
    otherwise add an empty space
    """
    if pvalue<0.01:
        adddot = '**' # significance marker
    elif pvalue<0.05:
        adddot = '*'# add significance marker
    else:
        adddot = ' ' # no significance
    return adddot

## Import burned area data
from `Data_CreateModelData`

In [9]:
# import observed burned area
filename = data_string + 'WUMI-ecoprovinces'
wumi = pd.read_csv(filename+'_all.txt').set_index('Unnamed: 0')
wumifor = pd.read_csv(filename+'_for.txt').set_index('Unnamed: 0')
wuminon = pd.read_csv(filename+'_non.txt').set_index('Unnamed: 0')
print('imported '+filename+' all, for, non')

imported data//WUMI-ecoprovinces all, for, non


## Import observed SST gradient data
from `Data_CreateENSOIndex` and `Data_CreateModelData`

In [10]:
# import observed SST gradient
filename = data_string + 'sstgrad_seasons_' + climindname+'_82_y.txt'
climind82_seasons = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import de-trended SST gradient
filename = data_string + 'sstgrad_seasons_' + climindnameDT+'_82_y.txt'
climind82_seasonsDT = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import observed SST gradient (with extra year, for comparison)
filename = data_string + 'sstgrad_seasons_' + climindname+'_81_y.txt'
climind81_seasons = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# IMPORT gSST AVERAGE (average of concurrent-season and prior-season)
# import observed SST gradient
filename = data_string + 'sstgrad_seasons_' + climindname+'_82_y_avgcurr-prior.txt'
climind82_seasonsavg = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import de-trended SST gradient
filename = data_string + 'sstgrad_seasons_' + climindnameDT+'_82_y_avgcurr-prior.txt'
climind82_seasonsDTavg = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# put the dataframes together
climind82_seasons = pd.concat([climind82_seasons, climind82_seasonsavg], axis=1)
climind82_seasonsDT = pd.concat([climind82_seasonsDT, climind82_seasonsDTavg], axis=1)

imported data//sstgrad_seasons_patch125-155_nino3-34_82_y.txt
imported data//sstgrad_seasons_DeTrend_patch125-155_nino3-34_82_y.txt
imported data//sstgrad_seasons_patch125-155_nino3-34_81_y.txt
imported data//sstgrad_seasons_patch125-155_nino3-34_82_y_avgcurr-prior.txt
imported data//sstgrad_seasons_DeTrend_patch125-155_nino3-34_82_y_avgcurr-prior.txt


## Import observed climate data
from `Data_CreateModelData`

In [11]:
# import OBSERVED climate data to feed into models
# climate is predicted using SST, but this is the climate observed data to check

dfframesall = {}
dfframesfor = {}
dfframesnon = {}

climind = pd.read_csv(data_string + 'gradient_'+climindname+'.txt', header=None, sep=",", skiprows=[0]).set_index(0)
for iecoreg in np.arange(len(province_num)):
    filename = data_string + 'climate_ecoprovinces_'
    print(dfnames[iecoreg])
    dfframesall[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_all.txt').set_index('Unnamed: 0')
    dfframesfor[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_for.txt').set_index('Unnamed: 0')
    dfframesnon[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_non.txt').set_index('Unnamed: 0')

allwestUS
ecoprov1
ecoprov2
ecoprov3
ecoprov4
ecoprov5
ecoprov6
ecoprov7
ecoprov8
ecoprov9
ecoprov10
ecoprov12
ecoprov13
ecoprov14
ecoprov15
ecoprov16
ecoprov17
ecoprov18
ecoprov19
ecoprov20
ecoprov21


## Import model outputs
from `Model_ENSOclim_Akaike`, `Model_Akaike`, and `Model_ENSOfull12-2022` (where I got the SST model, which doesn't change for these scenarios)

In [12]:
# import the climate under detrended gSST
# (already calculated as the 
# observed climate minus delta gSST contribution)

dfsstpredclimfor_sstdiff = {}
dfsstpredclimnon_sstdiff = {}
for thisname in dfnames:
    dictoutputnames = 'sstpred_for'
    filename = predict_string + 'Climate_'+thisname+'_'+dictoutputnames+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'.txt'
    dfsstpredclimfor_sstdiff[thisname] = pd.read_csv(filename).set_index('Unnamed: 0')
    dictoutputnames = 'sstpred_non'
    filename = predict_string + 'Climate_'+thisname+'_'+dictoutputnames+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'.txt'
    dfsstpredclimnon_sstdiff[thisname] = pd.read_csv(filename).set_index('Unnamed: 0')

In [13]:
# get the model input variable names from the txt files
# climate variables to predict burned area

with open(model_string + "modeloutput_burnarea_for.txt", "r") as f:
    inputsfor_clim = [line.strip() for line in f]
f.close()
with open(model_string + "modeloutput_burnarea_non.txt", "r") as f:
    inputsnon_clim = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
# get ecoregions so iteration is not manual
iinputsfor_clim = [i for i, e in 
               enumerate(inputsfor_clim) if "+++" in e]
# get ecoregions so iteration is not manual
iinputsnon_clim = [i for i, e in 
               enumerate(inputsnon_clim) if "+++" in e]
# add in last index
iinputsfor_clim.append(len(inputsfor_clim)+1)
iinputsnon_clim.append(len(inputsnon_clim)+1)

# Setup for exporting burned area predictions

Predicted burned area
* SST-predicted burned area (observed SST, goes through SST-clim and clim-BA models)
* climate-predicted burned area (observed climate, goes through clim-BA model)

In [14]:
# a bunch of dictionaries for storage

# SST-predicted, from climate_sstdiff
dictsstpredBAall = {}
dictsstpredBAfor = {}
dictsstpredBAnon = {}
# climate-predicted, from observed climate
dictclimpredBAall = {}
dictclimpredBAfor = {}
dictclimpredBAnon = {}

# Predicted Burned Area

Iterate through each ecoprovince by `ecoregname` and `iecoreg`, calculate the predicted burned area based on the observed climate `dfclimpredBA` and based on the SST difference `dfsstpredBA`.

Calculate full WUMI without SST gradient trend (DT scenario), outputted as `BurnArea_wumiDT`

In [15]:
# iterate through different experiments
# export to a dict by experiment name


for j, thisexperiment in enumerate(experimentnames):

    # parse experiment list
    thisexplistfor = experimentfor_list[j]
    thisexplistnon = experimentnon_list[j]

    # a bunch of dictionaries for storage
    # SST-predicted, from climate_sstdiff
    dfsstpredBAall = []
    dfsstpredBAfor = []
    dfsstpredBAnon = []
    # climate-predicted, from observed climate
    dfclimpredBAall = []
    dfclimpredBAfor = []
    dfclimpredBAnon = []
    
    # write the same print statements to this file
    modeloutputfile = 'Model_ENSOfull-outputsClim-BA_'+experimentnames[j]+'.txt'
    f = open(modeloutputfile, 'w') # print to this file
    
    # get this ecoregion
    for thisi, thisname in enumerate(dfnames):
        ecoregname = thisname
        iecoreg = thisi
    
        ###########
    
        landname = 'for'
        # finalclimpred is fed into the burned area model
        filename = model_string + 'model_'+landname+'_burnedarea_'+ecoregname+'.sav'
        modelburnarea = joblib.load(filename)
    
        # grab the adjusted climate prediction, predict area burned
        Xclimpred = dfsstpredclimfor_sstdiff[ecoregname]
        climnamessort = Xclimpred.columns
        # storage
        settozero = np.zeros(len(Xclimpred.columns.values))
        settozerosort = np.zeros(len(Xclimpred.columns.values))
        
        # iterate through each var name to figure out vars to hold constant
        for k, label in enumerate(climnamessort.values):
            tmplabeljustvar = label.split(' ')[0]
            tmplabeljustyr = label.split(' ')[1]
            tmpnameyrlabel = tmplabeljustvar + ' ' + tmplabeljustyr
            # set to one if it's a variable to hold constant
            if tmpnameyrlabel in thisexplistfor:
                # hold it constant
                settozero[k] = 1
        # set the model coefficients to zero using settozero
        tmpcoef = modelburnarea.coef_
        tmpcoef[settozero.astype(bool)] = 0
        modelburnarea.coef_ = tmpcoef # REPLACE coefs in MODEL
        
        # predict area burned for detrended sst scenario
        burnareapred = modelburnarea.predict(Xclimpred) # predicted value
    
        # grab the observed data, predict area burned
        Xclimpredobs = dfframesfor[ecoregname][climnamessort]
        print(prov_abbr_names[thisi] + ' - FOREST BA-model vars: ')
        f.write(prov_abbr_names[thisi] + ' - FOREST: \n')
        tmpvariableeq = dfframesfor[ecoregname][climnamessort].columns.values
        print('log10(BA) = ')
        f.write('log10(BA) = \n')
        for thisvari, thisvarname in enumerate(tmpvariableeq):
            tmpfullname, thisunits, thiscolor = suppclimplotformat(thisvarname)
            print('+ {:.3f}({:}) '.format(modelburnarea.coef_[thisvari], tmpfullname))
            f.write('+ {:.3f}({:}) '.format(modelburnarea.coef_[thisvari], tmpfullname))
        print('+ {:.3f} + \u03B5\n'.format(modelburnarea.intercept_))
        f.write('+ {:.3f} + \u03B5\n'.format(modelburnarea.intercept_))
        burnareapredobs = modelburnarea.predict(Xclimpredobs)
    
        # SAVE the predicted outputs
        dfsstpredBAfor.append(burnareapred) # sst-predicted
        dfclimpredBAfor.append(burnareapredobs) # clim-predicted
    
        # all data
        wumicol = np.array(wumifor.iloc[:,iecoreg])
        fig, ax = plt.subplots(figsize=(6,3))
        yearxaxis = np.arange(firstyear, finalyear+1)
        ax.plot(yearxaxis, wumicol, c='k', label='Observed burned area')
        ax.plot(yearxaxis, 10**burnareapredobs, label='From climate, observed')
        ax.plot(yearxaxis, 10**burnareapred, label='From climate, no SST trend')
        ax.legend()
        tmpcorr = pearsonr(burnareapredobs, burnareapred)
        ax.set_title('Burned area in '+landname + ', '+ ecoregname+ ', \nObserved Climate vs. Climate under '+ climindnameDT);
        fig.tight_layout()
        #plt.savefig(figfolder + 'Model_ENSOfull12-2022_'+landname+'_'+ecoregname+'_BurnAreaPred')
        plt.close()
        
        ###########
    
        landname = 'non'
    
        # finalclimpred is fed into the burned area model
        filename = model_string + 'model_'+landname+'_burnedarea_'+ecoregname+'.sav'
        modelburnarea = joblib.load(filename)
    
        # grab the adjusted climate prediction, predict area burned
        Xclimpred = dfsstpredclimnon_sstdiff[ecoregname]
        climnamessort = Xclimpred.columns
        # storage
        settozero = np.zeros(len(Xclimpred.columns.values))
        settozerosort = np.zeros(len(Xclimpred.columns.values))
        
        # iterate through each var name to figure out vars to hold constant
        for k, label in enumerate(climnamessort.values):
            tmplabeljustvar = label.split(' ')[0]
            tmplabeljustyr = label.split(' ')[1]
            tmpnameyrlabel = tmplabeljustvar + ' ' + tmplabeljustyr
            # set to one if it's a variable to hold constant
            if tmpnameyrlabel in thisexplistnon:
                # hold it constant
                settozero[k] = 1
        # set the model coefficients to zero using settozero
        tmpcoef = modelburnarea.coef_
        tmpcoef[settozero.astype(bool)] = 0
        modelburnarea.coef_ = tmpcoef # REPLACE coefs in MODEL
        
        # predict area burned for detrended sst scenario
        burnareapred = modelburnarea.predict(Xclimpred) # predicted value
    
        # grab the observed data, predict area burned
        Xclimpredobs = dfframesnon[ecoregname][climnamessort]
        print(prov_abbr_names[thisi] + ' - NON-FOREST BA-model: ')
        f.write(prov_abbr_names[thisi] + ' - NON-FOREST:\n')
        tmpvariableeq = dfframesnon[ecoregname][climnamessort].columns.values
        print('log10(BA) = ')
        f.write('log10(BA) = \n')
        for thisvari, thisvarname in enumerate(tmpvariableeq):
            tmpfullname, thisunits, thiscolor = suppclimplotformat(thisvarname)
            print('+ {:.3f}({:}) '.format(modelburnarea.coef_[thisvari], tmpfullname))
            f.write('+ {:.3f}({:}) '.format(modelburnarea.coef_[thisvari], tmpfullname))
    
        print('+ {:.3f} + \u03B5\n'.format(modelburnarea.intercept_))
        f.write('+ {:.3f} + \u03B5\n'.format(modelburnarea.intercept_))
        burnareapredobs = modelburnarea.predict(Xclimpredobs)
    
        # SAVE the predicted outputs
        dfsstpredBAnon.append(burnareapred) # sst-predicted
        dfclimpredBAnon.append(burnareapredobs) # clim-predicted
    
        # all data
        wumicol = np.array(wuminon.iloc[:,iecoreg])
        fig, ax = plt.subplots(figsize=(6,3))
        yearxaxis = np.arange(firstyear, finalyear+1)
        ax.plot(yearxaxis, wumicol, c='k', label='Observed burned area')
        ax.plot(yearxaxis, 10**burnareapredobs, label='From climate, observed')
        ax.plot(yearxaxis, 10**burnareapred, label='From climate, no SST trend')
        ax.legend()
        tmpcorr = pearsonr(burnareapredobs, burnareapred)
        ax.set_title('Burned area in '+landname + ', '+ ecoregname+ ', \nObserved Climate vs. Climate under '+ climindnameDT);
        fig.tight_layout()
        #plt.savefig(figfolder + 'Model_ENSOfull12-2022_'+landname+'_'+ecoregname+'_BurnAreaPred')
        plt.close()
        ############
    
        landname = 'all'
        
        # instead of using model, we sum forest+nonforest predictions
        burnareapred = np.log10((10**dfsstpredBAfor[-1])+(10**dfsstpredBAnon[-1]))
        burnareapredobs = np.log10((10**dfclimpredBAfor[-1])+(10**dfclimpredBAnon[-1]))
        # SAVE the predicted outputs
        dfsstpredBAall.append(burnareapred) # sst-predicted
        dfclimpredBAall.append(burnareapredobs) # clim-predicted
    
        # all data
        wumicol = np.array(wumi.iloc[:,iecoreg])
        fig, ax = plt.subplots(figsize=(6,3))
        yearxaxis = np.arange(firstyear, finalyear+1)
        ax.plot(yearxaxis, wumicol, c='k', label='Observed burned area')
        ax.plot(yearxaxis, 10**burnareapredobs, label='From climate, observed')
        ax.plot(yearxaxis, 10**burnareapred, label='From climate, no SST trend')
        ax.legend()
        tmpcorr = pearsonr(burnareapredobs, burnareapred)
        ax.set_title('Burned area in '+landname + ', '+ ecoregname+ ', \nObserved Climate vs. Climate under '+ climindnameDT);
        fig.tight_layout()
        #plt.savefig(figfolder + 'Model_ENSOfull12-2022_'+'for+non'+'_'+ecoregname+'_BurnAreaPred')
        plt.close()
    
    # close file
    f.close()
    # save the burn areas to the dict version
    # detrended-SST version
    dictsstpredBAall[thisexperiment] = dfsstpredBAall
    dictsstpredBAfor[thisexperiment] = dfsstpredBAfor
    dictsstpredBAnon[thisexperiment] = dfsstpredBAnon
    # climate-predicted, from observed climate
    dictclimpredBAall[thisexperiment] = dfclimpredBAall
    dictclimpredBAfor[thisexperiment] = dfclimpredBAfor
    dictclimpredBAnon[thisexperiment] = dfclimpredBAnon

0: All western US - FOREST BA-model vars: 
log10(BA) = 
+ 0.287(VPD y-0,JAS) 
+ 0.207(VPD y-0,AMJ) 
+ 0.112(Tmax y-0,OND) 
+ 3.392 + ε

0: All western US - NON-FOREST BA-model: 
log10(BA) = 
+ 0.150(VPD y-0,JAS) 
+ 0.000(Precipitation y-1,AMJ) 
+ 3.695 + ε

1: American Semi-Desert - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-1,JFM) 
+ 1.103 + ε

1: American Semi-Desert - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Precipitation y-1,OND) 
+ 0.000(Precipitation y-1,JFM) 
+ 0.115(Tmax y-0,JAS) 
+ 2.038 + ε

2: AZ-NM Mountains - FOREST BA-model vars: 
log10(BA) = 
+ 0.262(VPD y-0,AMJ) 
+ 0.245(Tmax y-0,JAS) 
+ 2.284 + ε

2: AZ-NM Mountains - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Wind y-0,JFM) 
+ 0.814(Tmax y-0,JAS) 
+ 0.000(Wet_days y-0,JAS) 
+ 0.000(Precipitation y-1,AMJ) 
+ 1.431 + ε

x: Black Hills Coniferous Forest - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-0,AMJ) 
+ 0.322 + ε



/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warning

x: Black Hills Coniferous Forest - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Precipitation y-1,JFM) 
+ -1.234 + ε

3: CA Coast Chapparral Forest - FOREST BA-model vars: 
log10(BA) = 
+ 0.347(VPD y-0,OND) 
+ 0.154 + ε

3: CA Coast Chapparral Forest - NON-FOREST BA-model: 
log10(BA) = 
+ 0.311(VPD y-0,OND) 
+ 1.846 + ε

4: CA Coast Range Open Woodland - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-0,OND) 
+ 0.000(Wet_days y-0,JFM) 
+ 0.298(VPD y-0,JAS) 
+ 0.946 + ε

4: CA Coast Range Open Woodland - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Precipitation y-1,JFM) 
+ 0.152(VPD y-0,JAS) 
+ 2.430 + ε

x: California Coastal Steppe-Redwood - FOREST BA-model vars: 
log10(BA) = 
+ 0.552(Tmax y-0,JAS) 
+ -0.701 + ε

x: California Coastal Steppe-Redwood - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ -1.234 + ε

5: CA Dry Steppe Province - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-1,OND) 
+ -2.653 + ε

5: CA Dry Steppe Province - NON-FOREST BA-model

/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warning

8: CO Plateau - FOREST BA-model vars: 
log10(BA) = 
+ 0.408(VPD y-0,AMJ) 
+ 0.308(Tmax y-0,OND) 
+ 1.761 + ε

8: CO Plateau - NON-FOREST BA-model: 
log10(BA) = 
+ 0.225(Tmax y-0,OND) 
+ 0.000(Solar_radiation y-0,JAS) 
+ 1.852 + ε

9: Great Plains - FOREST BA-model vars: 
log10(BA) = 
+ 0.361(VPD y-0,JAS) 
+ 0.269(Tmax y-0,AMJ) 
+ 0.000(Precipitation y-1,JFM) 
+ 1.681 + ε

9: Great Plains - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Wet_days y-0,JAS) 
+ 0.000(Precipitation y-1,AMJ) 
+ 2.468 + ε

10: IM Semi-Desert - FOREST BA-model vars: 
log10(BA) = 
+ 0.258(VPD y-0,JAS) 
+ 0.000(Wet_days y-0,AMJ) 
+ 1.529 + ε

10: IM Semi-Desert - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Precipitation y-1,AMJ) 
+ 0.311(VPD y-0,JAS) 
+ 0.000(Solar_radiation y-0,JFM) 
+ 0.000(Precipitation y-1,JAS) 
+ 0.000(Wind y-0,JAS) 
+ 3.156 + ε

11: IM Semi-Desert and Desert - FOREST BA-model vars: 
log10(BA) = 
+ 0.301(VPD y-0,AMJ) 
+ 0.000(Wind y-0,JFM) 
+ 0.180(VPD y-0,JAS) 
+ 1.781 + ε

11: IM Semi-Desert an

...

9: Great Plains - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(Tmax y-0,AMJ) 
+ 0.000(Precipitation y-1,JFM) 
+ 1.681 + ε

9: Great Plains - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Wet_days y-0,JAS) 
+ 0.000(Precipitation y-1,AMJ) 
+ 2.468 + ε

10: IM Semi-Desert - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(Wet_days y-0,AMJ) 
+ 1.529 + ε

10: IM Semi-Desert - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Precipitation y-1,AMJ) 
+ 0.000(VPD y-0,JAS) 
+ 0.000(Solar_radiation y-0,JFM) 
+ 0.000(Precipitation y-1,JAS) 
+ 0.000(Wind y-0,JAS) 
+ 3.156 + ε

11: IM Semi-Desert and Desert - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,AMJ) 
+ 0.000(Wind y-0,JFM) 
+ 0.000(VPD y-0,JAS) 
+ 1.781 + ε

11: IM Semi-Desert and Desert - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Precipitation y-1,JFM) 
+ 0.000(Wind y-0,JFM) 
+ 2.585 + ε

12: Middle Rocky Mountain Steppe - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(Tmax y-0,AMJ) 
+

/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.

13: NV-UT Mountains - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Precipitation y-1,AMJ) 
+ 0.000(Tmax y-0,AMJ) 
+ 0.000(Wind y-0,JFM) 
+ 1.691 + ε

14: Northern Rocky Mountain Forest - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(VPD y-0,AMJ) 
+ 1.825 + ε

14: Northern Rocky Mountain Forest - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(Precipitation y-1,JAS) 
+ 0.000(VPD y-0,JFM) 
+ 1.430 + ε

x: Pacific Lowland Mixed Forest Province - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(Tmax y-0,OND) 
+ -0.416 + ε



/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not de

x: Pacific Lowland Mixed Forest Province - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Wet_days y-0,JAS) 
+ -1.049 + ε

15: Sierran Steppe - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(Tmax y-0,AMJ) 
+ 0.000(Precipitation y-1,JFM) 
+ 0.000(VPD y-0,OND) 
+ 2.595 + ε

15: Sierran Steppe - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Precipitation y-1,JFM) 
+ 0.000(VPD y-0,AMJ) 
+ 0.000(VPD y-0,OND) 
+ 2.251 + ε

16: Southern Rocky Mountain Steppe - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(Solar_radiation y-0,JFM) 
+ 0.000(Wind y-0,JFM) 
+ 2.093 + ε

16: Southern Rocky Mountain Steppe - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(VPD y-0,AMJ) 
+ 0.000(Solar_radiation y-0,JAS) 
+ 1.770 + ε

17: SW Plateau and Plains - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(Tmax y-0,JFM) 
+ -1.459 + ε

17: SW Plateau and Plains - NON-FOREST BA-model: 
log10(BA) = 
+ 0.000(Precipitation y-1,JAS) 
+ 0.000(Wind y-0,JFM) 
+ 0.000(Precipitation y-1,JFM) 
+ 0.000(So

In [16]:
# set Ecoprovince 0 (all west US) as sum of the ecoprovinces

# iterate through each experiment to save the data
for j, thisexperiment in enumerate(experimentnames):
    print('output for experiment ['+thisexperiment+']')
    dfsstpredBAall = dictsstpredBAall[thisexperiment]
    dfsstpredBAfor = dictsstpredBAfor[thisexperiment]
    dfsstpredBAnon = dictsstpredBAnon[thisexperiment]
    # climate-predicted, from observed climate
    dfclimpredBAall = dictclimpredBAall[thisexperiment]
    dfclimpredBAfor = dictclimpredBAfor[thisexperiment]
    dfclimpredBAnon = dictclimpredBAnon[thisexperiment]
    
    # make everything unlogged
    # then sum the burned area from all ecoprovinces
    # then log it again, and set the first ecoprovince as the sums
    # all
    tmpburnarea = [exponentialit(i) for i in dfsstpredBAall]
    dfsstpredBAall[0] = np.log10(sum(tmpburnarea[1:]))
    tmpburnarea = [exponentialit(i) for i in dfclimpredBAall]
    dfclimpredBAall[0] = np.log10(sum(tmpburnarea[1:]))
    # for
    tmpburnarea = [exponentialit(i) for i in dfsstpredBAfor]
    dfsstpredBAfor[0] = np.log10(sum(tmpburnarea[1:]))
    tmpburnarea = [exponentialit(i) for i in dfclimpredBAfor]
    dfclimpredBAfor[0] = np.log10(sum(tmpburnarea[1:]))
    # non
    tmpburnarea = [exponentialit(i) for i in dfsstpredBAnon]
    dfsstpredBAnon[0] = np.log10(sum(tmpburnarea[1:]))
    tmpburnarea = [exponentialit(i) for i in dfclimpredBAnon]
    dfclimpredBAnon[0] = np.log10(sum(tmpburnarea[1:]))
    
    
    # calculate the full WUMI without the SST gradient trend
    # observed BA - (observed SST in clim-BA model BA - DT SST clim-BA model BA)
    
    # rearrange for pandas df
    dfsstpredBAall = np.transpose(np.stack(dfsstpredBAall))
    dfsstpredBAfor = np.transpose(np.stack(dfsstpredBAfor))
    dfsstpredBAnon = np.transpose(np.stack(dfsstpredBAnon))
    dfclimpredBAall = np.transpose(np.stack(dfclimpredBAall))
    dfclimpredBAfor = np.transpose(np.stack(dfclimpredBAfor))
    dfclimpredBAnon = np.transpose(np.stack(dfclimpredBAnon))

    
    
    dictoutputs = [dfsstpredBAall, dfsstpredBAfor, dfsstpredBAnon,
                   dfclimpredBAall, dfclimpredBAfor, dfclimpredBAnon]
    dictoutputnames = ['sstpred_all', 'sstpred_for', 'sstpred_non',
                       'climpred_all','climpred_for','climpred_non']
    
    # convert to pandas df and export
    for i,thisdict in enumerate(dictoutputs):
        tmpoutput = pd.DataFrame(thisdict, columns=[dfnames], index=yearxaxis)
        if i<3: # export the SST-predicted
            filename = predict_string + 'BurnArea_'+dictoutputnames[i]+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment+'.txt'
            tmpoutput.to_csv(filename)
        if (i>=3) & (i<6): # export the climate-predicted
            filename = predict_string + 'BurnArea_'+dictoutputnames[i]+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment+'.txt'
            tmpoutput.to_csv(filename)
        print('Exported: '+filename)

output for experiment [warmingvarsonly]
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_all_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_for_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_non_1984-2022_warmingvarsonly.txt
output for experiment [prioryrwettingonly]
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_prioryrwettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3

# Ensemble Error

Replaces the current `wumifor`, `wuminon`, `dfclimpredBAfor`, and `dfclimpredBAnon`

September 4, 2024

Because our model predicts log10(BA), it is prone to errors at higher burned area values. In general, by predicting log10(BA), our model will tend to underestimate . When we take the mean of log10(BA) for the entire timeseries, then the mean would be an underestimate of the actual mean. (the mean of 10^1, 10^2, and 10^3 would be (1+2+3)/3, which is 2, but that would be 100. But the real average would be 370.

* BA_mod = modeled burned area
* BA_obs = observed burned area

Method is as follows:
1. Calculate the time series of errors `BA_error_log10 = log10(BA_obs) - log10(BA_mod)`
3. make an ensemble of 500 time series of `log10(BA_mod_plus_error)`.
    * for each step in the time series of `BA_mod`, the error is randomly selected and added to the step in the time series by sampling from the time series of `BA_error_log10` with replacement, not including sampling from the index year being selected for.
    * Result is one 39x500 matrix (years x ensembles) of randomly-selected years is created, and used to pick for all of the ecoprovinces and timeseries at once.
4. Un-log all 500 time series of log10(BA_mod_plus_error) to get the 500-member ensemble in original km^2 units.
5. Convert negative values to 0 km^2
6. Take the average of all 500 time series.

In [17]:
# import OBSERVED burned area wumi
filename = data_string + 'WUMI-ecoprovinces'
wumi = pd.read_csv(filename+'_all.txt').set_index('Unnamed: 0')
wumifor = pd.read_csv(filename+'_for.txt').set_index('Unnamed: 0')
wuminon = pd.read_csv(filename+'_non.txt').set_index('Unnamed: 0')

# import DETRENDED burned area wumi
filename = predict_string + 'BurnArea_wumiDT'
filenameend = '_'+str(firstyear)+'-'+str(finalyear)+'.txt'
wumiDT = pd.read_csv(filename+'_all_'+climfilename+filenameend).set_index('Unnamed: 0')
wumiforDT = pd.read_csv(filename+'_for_'+climfilename+filenameend).set_index('Unnamed: 0')
wuminonDT = pd.read_csv(filename+'_non_'+climfilename+filenameend).set_index('Unnamed: 0')

# PREDICTED BURNED AREA from observed SST and observed climate 
# (Model_ENSOfull12-2022 must be updated)

#filename = predict_string + 'BurnArea_'
filename = predict_string + 'BurnArea_'

# predicted burned area from observed climate
#BAclimpredall = pd.read_csv(filename+'climpred_all_'+str(firstyear)+'-'+str(finalyear)+'.txt').set_index('Unnamed: 0')
BAclimpredfor = pd.read_csv(filename+'climpred_for_'+str(firstyear)+'-'+str(finalyear)+'.txt').set_index('Unnamed: 0')
BAclimprednon = pd.read_csv(filename+'climpred_non_'+str(firstyear)+'-'+str(finalyear)+'.txt').set_index('Unnamed: 0')

# predicted burned area removing SST gradient trend
# (observed climate - climate without effect of SST trend)
#BAclimpredall_DT = pd.read_csv(filename+'sstpred_all_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'.txt').set_index('Unnamed: 0')
BAclimpredfor_DT = pd.read_csv(filename+'sstpred_for_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'.txt').set_index('Unnamed: 0')
BAclimprednon_DT = pd.read_csv(filename+'sstpred_non_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'.txt').set_index('Unnamed: 0')

In [18]:
# log all wumi values
# BAclimpred is already log
BA_obsfor = np.ma.masked_invalid(np.log10(wumifor)).filled(0)
BA_obsnon = np.ma.masked_invalid(np.log10(wuminon)).filled(0)
BA_obsfor = pd.DataFrame(BA_obsfor, index=yearxaxis, columns=dfnames)
BA_obsnon = pd.DataFrame(BA_obsnon, index=yearxaxis, columns=dfnames)

# get log difference BA_obs - BA_model
BA_errorfor_log10 = BA_obsfor - BAclimpredfor
BA_errornon_log10 = BA_obsnon - BAclimprednon

## MAKE ERROR MATRIX FROM THE OBSERVED VS. FULLY-CALIBRATED MODEL

# iterate through each ecoprovince
nensemble = 500 # number of ensemble members

# first make matrix of random years (n_years x n_ensemble members)
# RULE: 
istop = 0
iyears = np.arange(0,len(yearxaxis))
errormatrix = np.empty((0,len(yearxaxis)))
while istop < (nensemble):
    tmprandom = np.array([np.random.choice(np.delete(iyears, i)) for i in range(len(iyears))])
    # randomly select errors and construct a timeseries
    tmprandom = tmprandom.reshape(1, -1)
    errormatrix = np.concatenate([errormatrix, 
                                  tmprandom], axis=0)
    istop+=1

/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/internals/blocks.py:366: RuntimeWarning: divide by zero encountered in log10
  result = func(self.values, **kwargs)


In [19]:
# import the BA predicted from the scenarios, arranged 
# into the dict format

dictsstpredBAall = {}
dictsstpredBAfor = {}
dictsstpredBAnon = {}
dictclimpredBAall = {}
dictclimpredBAfor = {}
dictclimpredBAnon = {}

for j, thisexperiment in enumerate(experimentnames):
    print('reimport experiment ['+thisexperiment+']')
    dictoutputnames = ['sstpred_all', 'sstpred_for', 'sstpred_non',
                       'climpred_all','climpred_for','climpred_non']
    filename = predict_string + 'BurnArea_sstpred_all_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    dictsstpredBAall[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')
    filename = predict_string + 'BurnArea_sstpred_for_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    dictsstpredBAfor[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')
    filename = predict_string + 'BurnArea_sstpred_non_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    dictsstpredBAnon[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')
    filename = predict_string + 'BurnArea_climpred_all_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    dictclimpredBAall[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')
    filename = predict_string + 'BurnArea_climpred_for_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    dictclimpredBAfor[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')
    filename = predict_string + 'BurnArea_climpred_non_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    dictclimpredBAnon[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')

reimport experiment [warmingvarsonly]
reimport experiment [prioryrwettingonly]
reimport experiment [y0wettingonly]


In [20]:
# ITERATE THROUGH THE EXPERIMENTS AND APPLY ERROR MATRIX

    # EXPORT CLIMATE-PREDICTED (SST TRENDED) AND
    # SST-PREDICTED (SST-DETRENDED) MODELED BA
    
    # for one ecoregion
    # pick ecoregion name, select column

for j,thisexperiment in enumerate(experimentnames):
    # import the experiment
    BAclimpredfor = dictclimpredBAfor[thisexperiment]
    BAclimprednon = dictclimpredBAnon[thisexperiment]
    BAclimpredfor_DT = dictsstpredBAfor[thisexperiment]
    BAclimprednon_DT = dictsstpredBAnon[thisexperiment]
    
    # storage for final
    BA_mod_plus_errorfor = np.empty((0,len(yearxaxis)))
    BA_mod_plus_errornon = np.empty((0,len(yearxaxis)))
    BA_modDT_plus_errorfor = np.empty((0,len(yearxaxis)))
    BA_modDT_plus_errornon = np.empty((0,len(yearxaxis)))
    
    for thisregion in dfnames:   
        # select the modeled BA
        tmpBAmodfor = BAclimpredfor.loc[:,thisregion] # trended SST
        tmpBAmodnon = BAclimprednon.loc[:,thisregion]
        tmpBAmodDTfor = BAclimpredfor_DT.loc[:,thisregion] # detrended SST
        tmpBAmodDTnon = BAclimprednon_DT.loc[:,thisregion]
    
        # select errors according to errormatrix
        tmperrorfor = BA_errorfor_log10.loc[:,thisregion] # error BA_obs-BA_mod(trended)
        tmperrornon = BA_errornon_log10.loc[:,thisregion]
        tmpBA_randomerrorfor = np.empty((0,len(yearxaxis)))
        tmpBA_randomerrornon = np.empty((0,len(yearxaxis)))
        # iterate through errormatrix
        for j in np.arange(0,nensemble):
            tmpcolerrorfor = tmperrorfor.iloc[errormatrix[j]].values.reshape(1, -1)
            tmpcolerrornon = tmperrornon.iloc[errormatrix[j]].values.reshape(1, -1)
            tmpBA_randomerrorfor = np.concatenate([tmpBA_randomerrorfor, 
                                                   tmpcolerrorfor], axis=0)
            tmpBA_randomerrornon = np.concatenate([tmpBA_randomerrornon, 
                                                   tmpcolerrornon], axis=0)
    
        # add BA_model to the errors, then unlog
        tmpBA_mod_plus_errorfor = 10**(tmpBAmodfor.values[:, np.newaxis] + tmpBA_randomerrorfor.T)
        tmpBA_mod_plus_errornon = 10**(tmpBAmodnon.values[:, np.newaxis] + tmpBA_randomerrornon.T)
        tmpBA_modDT_plus_errorfor = 10**(tmpBAmodDTfor.values[:, np.newaxis] + tmpBA_randomerrorfor.T)
        tmpBA_modDT_plus_errornon = 10**(tmpBAmodDTnon.values[:, np.newaxis] + tmpBA_randomerrornon.T)
        
        # create pd dataframe
        tmpBA_mod_plus_errorfor = pd.DataFrame(tmpBA_mod_plus_errorfor, index=yearxaxis)
        tmpBA_mod_plus_errornon = pd.DataFrame(tmpBA_mod_plus_errornon, index=yearxaxis)
        tmpBA_modDT_plus_errorfor = pd.DataFrame(tmpBA_modDT_plus_errorfor, index=yearxaxis)
        tmpBA_modDT_plus_errornon = pd.DataFrame(tmpBA_modDT_plus_errornon, index=yearxaxis)
    
        # RULE: set negative values to zero
        tmpBA_mod_plus_errorfor[tmpBA_mod_plus_errorfor<0] = 0
        tmpBA_mod_plus_errornon[tmpBA_mod_plus_errornon<0] = 0
        tmpBA_modDT_plus_errorfor[tmpBA_modDT_plus_errorfor<0] = 0
        tmpBA_modDT_plus_errornon[tmpBA_modDT_plus_errornon<0] = 0
        
        # take mean mod+error of all n columns
        tmpBA_mod_plus_errorfor1 = tmpBA_mod_plus_errorfor.mean(axis=1).values.reshape(1, -1)
        tmpBA_mod_plus_errornon1 = tmpBA_mod_plus_errornon.mean(axis=1).values.reshape(1, -1)
        tmpBA_modDT_plus_errorfor1 = tmpBA_modDT_plus_errorfor.mean(axis=1).values.reshape(1, -1)
        tmpBA_modDT_plus_errornon1 = tmpBA_modDT_plus_errornon.mean(axis=1).values.reshape(1, -1)
        
        # append to DF
        BA_mod_plus_errorfor = np.concatenate([BA_mod_plus_errorfor, 
                                               tmpBA_mod_plus_errorfor1], axis=0)
        BA_mod_plus_errornon = np.concatenate([BA_mod_plus_errornon, 
                                               tmpBA_mod_plus_errornon1], axis=0)
        BA_modDT_plus_errorfor = np.concatenate([BA_modDT_plus_errorfor, 
                                               tmpBA_modDT_plus_errorfor1], axis=0)
        BA_modDT_plus_errornon = np.concatenate([BA_modDT_plus_errornon, 
                                               tmpBA_modDT_plus_errornon1], axis=0)
    
    # convert final result to DF
    BA_mod_plus_errorfor = pd.DataFrame(BA_mod_plus_errorfor.T, 
                                        index=yearxaxis, columns=dfnames)
    BA_mod_plus_errornon = pd.DataFrame(BA_mod_plus_errornon.T,
                                        index=yearxaxis, columns=dfnames)
    BA_modDT_plus_errorfor = pd.DataFrame(BA_modDT_plus_errorfor.T, 
                                        index=yearxaxis, columns=dfnames)
    BA_modDT_plus_errornon = pd.DataFrame(BA_modDT_plus_errornon.T,
                                        index=yearxaxis, columns=dfnames)
    # replace all western US column with sum
    BA_mod_plus_errorfor['allwestUS'] = BA_mod_plus_errorfor.iloc[:,1:].sum(axis=1)
    BA_mod_plus_errornon['allwestUS'] = BA_mod_plus_errornon.iloc[:,1:].sum(axis=1)
    BA_modDT_plus_errorfor['allwestUS'] = BA_modDT_plus_errorfor.iloc[:,1:].sum(axis=1)
    BA_modDT_plus_errornon['allwestUS'] = BA_modDT_plus_errornon.iloc[:,1:].sum(axis=1)
    
    
    # prep for export, reassign to variable names
    dfclimpredBAall = np.log10(BA_mod_plus_errorfor+BA_mod_plus_errornon).values
    dfclimpredBAfor = np.log10(BA_mod_plus_errorfor).values
    dfclimpredBAnon = np.log10(BA_mod_plus_errornon).values
    
    dfsstpredBAall = np.log10(BA_modDT_plus_errorfor+BA_modDT_plus_errornon).values
    dfsstpredBAfor = np.log10(BA_modDT_plus_errorfor).values
    dfsstpredBAnon = np.log10(BA_modDT_plus_errornon).values

    # export back to the experimental versions dict
    # detrended-SST version
    dictsstpredBAall[thisexperiment] = dfsstpredBAall
    dictsstpredBAfor[thisexperiment] = dfsstpredBAfor
    dictsstpredBAnon[thisexperiment] = dfsstpredBAnon
    # climate-predicted, from observed climate
    dictclimpredBAall[thisexperiment] = dfclimpredBAall
    dictclimpredBAfor[thisexperiment] = dfclimpredBAfor
    dictclimpredBAnon[thisexperiment] = dfclimpredBAnon

In [21]:
# iterate through each experiment to save the data
for j, thisexperiment in enumerate(experimentnames):
    print('output for experiment ['+thisexperiment+']')
    dfsstpredBAall = dictsstpredBAall[thisexperiment]
    dfsstpredBAfor = dictsstpredBAfor[thisexperiment]
    dfsstpredBAnon = dictsstpredBAnon[thisexperiment]
    # climate-predicted, from observed climate
    dfclimpredBAall = dictclimpredBAall[thisexperiment]
    dfclimpredBAfor = dictclimpredBAfor[thisexperiment]
    dfclimpredBAnon = dictclimpredBAnon[thisexperiment]
    
    # export overwrite
    dictoutputs = [dfsstpredBAall, dfsstpredBAfor, dfsstpredBAnon,
                   dfclimpredBAall, dfclimpredBAfor, dfclimpredBAnon]
    dictoutputnames = ['sstpred_all', 'sstpred_for', 'sstpred_non',
                       'climpred_all','climpred_for','climpred_non']
    # convert to pandas df and export
    for i,thisdict in enumerate(dictoutputs):
        tmpoutput = pd.DataFrame(thisdict, columns=[dfnames], index=yearxaxis)
        if i<3: # export the SST-predicted
            filename = predict_string + 'BurnArea_'+dictoutputnames[i]+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment+'.txt'
            tmpoutput.to_csv(filename)
        if (i>=3) & (i<6): # export the climate-predicted
            filename = predict_string + 'BurnArea_'+dictoutputnames[i]+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment+'.txt'
            tmpoutput.to_csv(filename)
        print('Exported: '+filename)

output for experiment [warmingvarsonly]
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_all_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_for_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_non_1984-2022_warmingvarsonly.txt
output for experiment [prioryrwettingonly]
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_prioryrwettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3

In [22]:
# datetime object containing current date and time
from datetime import datetime
now = datetime.now()
# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print("Model last run =", dt_string)

Model last run = 20/02/2025 23:12:04
